# 02 - Data Understanding & Statistical Profiling

## 1. Objectives
- Perform comprehensive statistical profiling of all attributes.
- Identify data anomalies, zeros, negatives, and outliers.
- Quantify class imbalance in the target variable `Personal Loan`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

df = pd.read_csv(project_root / "data" / "raw" / "Bank_Personal_Loan_Modelling.csv")
print("Data loaded successfully. Shape:", df.shape)

Data loaded successfully. Shape: (5000, 14)


## 2. Descriptive Statistics

In [2]:
# Numerical summary statistics
desc_df = df.describe().T
desc_df["variance"] = df.var()
desc_df["skewness"] = df.skew()
desc_df["kurtosis"] = df.kurt()
display(desc_df)

,count,mean,std,min,25%,50%,75%,max,variance,skewness,kurtosis
ID,5000.0,2500.500000,1443.520003,1.0,1250.75,2500.5,3750.25,5000.0,2.083750e+06,0.000000,-1.200000
Age,5000.0,45.338400,11.463166,23.0,35.00,45.0,55.00,67.0,1.314042e+02,-0.029341,-1.153067
Experience,5000.0,20.104600,11.467954,-3.0,10.00,20.0,30.00,43.0,1.315140e+02,-0.026325,-1.121523
Income,5000.0,73.774200,46.033729,8.0,39.00,64.0,98.00,224.0,2.119104e+03,0.841339,-0.044244
ZIP Code,5000.0,93152.503000,2121.852197,9307.0,91911.00,93437.0,94608.00,96651.0,4.502257e+06,-12.500221,486.204272
Family,5000.0,2.396400,1.147663,1.0,1.00,2.0,3.00,4.0,1.317130e+00,0.155221,-1.403586
CCAvg,5000.0,1.937938,1.747659,0.0,0.70,1.5,2.50,10.0,3.054312e+00,1.598443,2.646706
Education,5000.0,1.881000,0.839869,1.0,1.00,2.0,3.00,3.0,7.053801e-01,0.227093,-1.546116
Mortgage,5000.0,56.498800,101.713802,0.0,0.00,0.0,101.00,635.0,1.034570e+04,2.104002,4.756797
Personal Loan,5000.0,0.096000,0.294621,0.0,0.00,0.0,0.00,1.0,8.680136e-02,2.743607,5.529589


## 3. Anomaly Detection: Negative Experience Values
In the `Experience` column, the minimum value is **-3**. Professional experience cannot be negative.
Let us examine the negative experience records and their relationship with customer age.

In [3]:
neg_exp = df[df["Experience"] < 0]
print(f"Total records with negative experience: {len(neg_exp)} ({len(neg_exp)/len(df):.2%})")
print("\nNegative experience breakdown by Age & Education:")
display(neg_exp.groupby(["Age", "Experience", "Education"]).size().reset_index(name="Count"))

Total records with negative experience: 52 (1.04%)

Negative experience breakdown by Age & Education:


,Age,Experience,Education,Count
0,23,-3,1,1
1,23,-3,2,1
2,23,-2,2,4
3,23,-1,1,6
4,24,-3,3,2
5,24,-2,1,4
6,24,-2,2,1
7,24,-2,3,4
8,24,-1,1,1
9,24,-1,2,5


## 4. Analysis of Zero Values
Features like `Mortgage`, `Securities Account`, and `CD Account` contain significant zero values. Let us quantify the proportion of zeros for each attribute.

In [4]:
zero_counts = (df == 0).sum()
zero_pct = (df == 0).mean() * 100
zero_df = pd.DataFrame({"Zero_Count": zero_counts, "Zero_Percentage": zero_pct})
display(zero_df.sort_values(by="Zero_Percentage", ascending=False))

,Zero_Count,Zero_Percentage
CD Account,4698,93.96
Personal Loan,4520,90.40
Securities Account,4478,89.56
CreditCard,3530,70.60
Mortgage,3462,69.24
Online,2016,40.32
CCAvg,106,2.12
Experience,66,1.32
Family,0,0.00
Age,0,0.00


## 5. Target Class Distribution Analysis
The target is `Personal Loan`. Let us examine the class distribution to assess imbalance.

In [5]:
target_counts = df["Personal Loan"].value_counts()
target_prop = df["Personal Loan"].value_counts(normalize=True) * 100

print(f"Class 0 (Did not accept loan): {target_counts[0]:,} ({target_prop[0]:.2f}%)")
print(f"Class 1 (Accepted loan):       {target_counts[1]:,} ({target_prop[1]:.2f}%)")
print(f"Imbalance Ratio:               {target_counts[0] / target_counts[1]:.2f} : 1")

Class 0 (Did not accept loan): 4,520 (90.40%)
Class 1 (Accepted loan):       480 (9.60%)
Imbalance Ratio:               9.42 : 1


## 6. Key Data Understanding Insights
1. **Class Imbalance:** Only 9.6% of customers accepted the personal loan. Pure accuracy is an inadequate metric; precision, recall, F1, and PR-AUC must be prioritized.
2. **Negative Experience Anomaly:** 52 rows have negative experience (min = -3). All occur at young ages (Age 23-29). This will be handled during data cleaning.
3. **Mortgage Sparsity:** Over 68% of customers have zero mortgage.
4. **CD Account Rarity:** Only 6.04% of customers have a CD account, but CD holders have a very strong historical propensity to take loans.